<a href="https://colab.research.google.com/github/AbdulUMSL/EENG-1108/blob/main/Hw6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import requests
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
from openai import OpenAI

# Initialize the client to point to your local Ollama server
# By default, Ollama runs on port 11434
client = OpenAI(
    base_url='http://localhost:11434/v1',
    api_key='ollama', # Required but ignored by Ollama
)

def summarize_website(url):
    # 1. Scrape the website content
    try:
        response = requests.get(url)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')

        # Remove script and style elements to clean the text
        for script_or_style in soup(["script", "style"]):
            script_or_style.decompose()

        # Extract text and limit length to avoid overwhelming the local context window
        text = soup.get_text(separator=' ')
        clean_text = " ".join(text.split())[:10000]

        title = soup.title.string if soup.title else "this website"

    except Exception as e:
        return f"Error fetching the website: {e}"

    # 2. Use Llama 3.2 to summarize
    system_prompt = "You are a helpful assistant that summarizes websites concisely."
    user_prompt = f"Please provide a bulleted summary of the following content from {title}:\n\n{clean_text}"

    response = client.chat.completions.create(
        model="llama3.2",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.7
    )

    return response.choices[0].message.content

# --- Execution ---
url_to_summarize = "https://example.com" # Replace with your target URL
summary = summarize_website(url_to_summarize)

display(Markdown(f"## Summary of {url_to_summarize}\n\n{summary}"))